# ◉ GPU Optimizations

## H100 architecture

![gpu](../_asset/gpu/1.jpg)

![gpu](../_asset/gpu/2.png)

## Compute VS Data flow

![gpu](../_asset/gpu/flow.png)

## GPU to GPU inference

![gpu](../_asset/gpu/3.png)

![gpu](../_asset/gpu/4.jpg)

![gpu](../_asset/gpu/5.jpg)

How `152,384` and `0.0003456` are represented in different GPU number formats

| Format       |           Total Bits |                 Exponent Bits |                         Bias |                              Mantissa Bits |                  Approx Precision |                                     `152,384` |                                   `0.0003456` |
| ------------ | -------------------: | ----------------------------: | ---------------------------: | -----------------------------------------: | --------------------------------: | --------------------------------------------: | --------------------------------------------: |
| **FP32**     |                   32 |                             8 |                          127 |                                         23 |               ~7.2 decimal digits |                      `152,384`, exact integer |                       `0.0003456000122241676` |
| **FP16**     |                   16 |                             5 |                           15 |                                         10 |               ~3.3 decimal digits |         `+inf`, overflow; max finite `65,504` |                       `0.0003457069396972656` |
| **BF16**     |                   16 |                             8 |                          127 |                                          7 |               ~2.4 decimal digits |                                     `152,576` |                       `0.0003452301025390625` |
| **FP8 E4M3** |                    8 |                             4 |                            7 |                                          3 |               ~1.2 decimal digits |               out of range; max finite `±448` |                             `0.0`, underflows |
| **FP8 E5M2** |                    8 |                             5 |                           15 |                                          2 |               ~0.9 decimal digits |            out of range; max finite `±57,344` |                             `0.0003662109375` |
| **Raw INT8** |                    8 |                          none |                         none |                                       none |                      integer only |                   overflow if raw signed INT8 |                `0` if directly cast/truncated |


![gpu](../_asset/gpu/6.jpg)

## Operation and dispatch decision: CPU vs GPU

In [ ]:
import torch
torch.compiler.config.force_disable_caches = True

print("torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("capability:", torch.cuda.get_device_capability(0))
    print("CUDA (torch build):", torch.version.cuda)

import triton
print("triton version:", triton.__version__)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("using device:", device)

torch version: 2.11.0+cu128
CUDA available: True
device: NVIDIA GeForce RTX 5090 Laptop GPU
capability: (12, 0)
CUDA (torch build): 12.8
triton version: 3.6.0
using device: cuda


In [2]:
from torch.utils._python_dispatch import TorchDispatchMode

class LogDispatch(TorchDispatchMode):
    def __torch_dispatch__(self, func, types, args=(), kwargs=None):
        print(f"  -> dispatched to {func}")
        return func(*args, **(kwargs or {}))

x, y = torch.randn(4), torch.randn(4)

print("x + y:")
with LogDispatch():
    _ = x + y

print("torch.add(x, y):")
with LogDispatch():
    _ = torch.add(x, y)

print("x.add(y):")
with LogDispatch():
    _ = x.add(y)

print("x * 2:")
with LogDispatch():
    _ = x * 2

print("torch.mul(x, 2):")
with LogDispatch():
    _ = torch.mul(x, 2)

x + y:


  -> dispatched to aten.add.Tensor
torch.add(x, y):
  -> dispatched to aten.add.Tensor
x.add(y):
  -> dispatched to aten.add.Tensor
x * 2:
  -> dispatched to aten.mul.Tensor
torch.mul(x, 2):
  -> dispatched to aten.mul.Tensor


All five calls above resolve to just two distinct operators
(`aten.add.Tensor`, `aten.mul.Tensor`) regardless of which Python spelling
was used. That's one dispatch decision -- *which operator* -- and it's
purely syntactic.

### Dispatch decision

Every tensor has a `.device` (`cpu` or `cuda:0`, set when it's
created -- `torch.randn(..., device=...)` -- or moved with
`.to(device)`/`.cuda()`). PyTorch's dispatcher looks at the device (and
dtype) of an op's tensor arguments and routes the call to the matching
registered backend implementation -- a CPU tensor's `+` runs a vectorized
CPU loop; a CUDA tensor's `+` launches an actual CUDA kernel

In [3]:
x_cpu, y_cpu = torch.randn(4), torch.randn(4)
print("x_cpu + y_cpu -> ran on:", (x_cpu + y_cpu).device)

if device == "cuda":
    x_gpu, y_gpu = x_cpu.to("cuda"), y_cpu.to("cuda")
    print("x_gpu + y_gpu -> ran on:", (x_gpu + y_gpu).device,
          "(identical Python line, different physical kernel)")

    try:
        x_cpu + x_gpu
    except RuntimeError as e:
        print("mixing devices ->", e)
else:
    print("no CUDA device available in this environment to demonstrate the CPU/CUDA contrast directly.")

x_cpu + y_cpu -> ran on: cpu
x_gpu + y_gpu -> ran on: cuda:0 (identical Python line, different physical kernel)
mixing devices -> Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!


That covers *which* kernel a line of code calls. The next question is
*when* it runs relative to the rest of your program -- immediately, one
op at a time (eager mode), or gathered up and optimized as a whole
(`torch.compile`). That's where kernel launch overhead and fusion
actually come from.

## GPU Kernels: Eager vs Fused operation
![gpu](../_asset/gpu/kernel_1.png)

In [ ]:
def elementwise_chain(x):
    a = x * 2
    b = a + 1
    c = torch.sin(b)
    d = c * a
    return d.relu()

x = torch.randn(4096, 4096, device=device)

compiled_fn = torch.compile(elementwise_chain)

for f in (elementwise_chain, compiled_fn):
    f(x)
if device == "cuda":
    torch.cuda.synchronize()

/home/user/anaconda3/envs/pe_test/lib/python3.12/site-packages/torch/_dynamo/pgo.py:538: UserWarning: dynamo_pgo force disabled by torch.compiler.config.force_disable_caches
  warn_once(


In [5]:
import time

def bench(f, x, iters=200):
    if device == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters):
        f(x)
    if device == "cuda":
        torch.cuda.synchronize()
    return (time.perf_counter() - t0) / iters * 1e6  # microseconds/call

eager_us = bench(elementwise_chain, x)
compiled_us = bench(compiled_fn, x)
print(f"eager:    {eager_us:8.1f} us/call")
print(f"compiled: {compiled_us:8.1f} us/call")
print(f"speedup:  {eager_us / compiled_us:5.2f}x")

eager:      1182.7 us/call
compiled:    195.2 us/call
speedup:   6.06x


![gpu](../_asset/gpu/kernel_2.png)

## I/O challenges with Attention

![gpu](../_asset/gpu/att1.png)

![gpu](../_asset/gpu/att2.jpg)

![gpu](../_asset/gpu/att3.png)

![gpu](../_asset/gpu/att4.png)

## Flash Attention

TBD...

## TorchDynamo: bytecode capture and graphs

`torch._dynamo.explain` runs this process and reports what happened
without needing to actually execute a full compiled run.

In [6]:
import torch._dynamo as dynamo

def f(x, y):
    z = x + y
    z = z * 2
    return z.relu()

a, b = torch.randn(8, device=device), torch.randn(8, device=device)
explanation = dynamo.explain(f)(a, b)
print(explanation)

Graph Count: 1
Graph Break Count: 0
Op Count: 2
Break Reasons:
Ops per Graph:
  Ops 1:
    <built-in function add>
    <built-in function mul>
Out Guards:
  Guard 1:
    Name: ''
    Source: shape_env
    Create Function: SHAPE_ENV
    Guard Types: None
    Code List: None
    Object Weakref: None
    Guarded Class Weakref: None
  Guard 2:
    Name: ''
    Source: global
    Create Function: DETERMINISTIC_ALGORITHMS
    Guard Types: None
    Code List: None
    Object Weakref: None
    Guarded Class Weakref: None
  Guard 3:
    Name: ''
    Source: global
    Create Function: GRAD_MODE
    Guard Types: None
    Code List: None
    Object Weakref: None
    Guarded Class Weakref: None
  Guard 4:
    Name: ''
    Source: global
    Create Function: DEFAULT_DEVICE
    Guard Types: ['DEFAULT_DEVICE']
    Code List: ['utils_device.CURRENT_DEVICE == None']
    Object Weakref: None
    Guarded Class Weakref: None
  Guard 5:
    Name: ''
    Source: global
    Create Function: GLOBAL_STATE
    

/home/user/anaconda3/envs/pe_test/lib/python3.12/site-packages/torch/_dynamo/pgo.py:538: UserWarning: dynamo_pgo force disabled by torch.compiler.config.force_disable_caches
  warn_once(


Look at `graph_count` (how many separate FX graphs Dynamo produced --
ideally 1) and `graph_break_count` (ideally 0). Next, let's look at the
*actual* captured FX graph, which is a real `torch.fx.GraphModule`.

In [7]:
def show_graph(gm, example_inputs):
    print(gm.graph)
    return gm  # required: a backend must return a callable

compiled_f = torch.compile(f, backend=show_graph)
_ = compiled_f(a, b)

graph():
    %l_x_ : torch.Tensor [num_users=1] = placeholder[target=L_x_]
    %l_y_ : torch.Tensor [num_users=1] = placeholder[target=L_y_]
    %z : [num_users=1] = call_function[target=operator.add](args = (%l_x_, %l_y_), kwargs = {})
    %z_1 : [num_users=1] = call_function[target=operator.mul](args = (%z, 2), kwargs = {})
    %relu : [num_users=1] = call_method[target=relu](args = (%z_1,), kwargs = {})
    return (relu,)


Every row is one FX node: `placeholder` for inputs, `call_function`/
`call_method` for an op, `output` for the return. Notice the ops are
**not** yet ATen operators here -- `target=operator.add`,
`target=operator.mul`, and `relu` via `call_method`, i.e. roughly the
level of the Python source (`+`, `*`, `.relu()`), not `aten.add.Tensor`.
That's Dynamo's own graph: it records *which tensor operations happened*,
but doesn't itself lower them into PyTorch's internal ATen operator set.

Real ATen ops (`torch.ops.aten.*`) show up one stage later, from
AOTAutograd -- Section 7 exports a graph for a small MLP and you'll see
`aten.addmm.default`, `aten.relu.default`, etc. there. Keep the two levels
straight: Dynamo's FX graph mirrors the Python-level tensor ops that were
called; AOTAutograd's graph is the ATen-level one that actually gets
decomposed and handed to Inductor.

You can get the same view (plus a lot more: guards, bytecode diffs,
recompile reasons) by setting `TORCH_LOGS`. This is the single most useful
debugging knob for `torch.compile` and we'll use it repeatedly.

In [ ]:
import torch._dynamo as dynamo
dynamo.reset()  # clear cached graphs so logging below shows a fresh compile

import torch._logging
torch._logging.set_logs(graph_code=True)

def g(x):
    return (x * 2 + 1).relu()

torch.compile(g)(torch.randn(8, device=device))

torch._logging.set_logs()  # reset so later cells aren't noisy

V0821 12:05:44.479000 371515 site-packages/torch/_dynamo/output_graph.py:2417] [0/0] [__graph_code] TRACED GRAPH
V0821 12:05:44.479000 371515 site-packages/torch/_dynamo/output_graph.py:2417] [0/0] [__graph_code]  ===== __compiled_fn_7_fe352b6b_09ff_4fc3_894f_f2092d735189 =====
V0821 12:05:44.479000 371515 site-packages/torch/_dynamo/output_graph.py:2417] [0/0] [__graph_code]  /home/user/anaconda3/envs/pe_test/lib/python3.12/site-packages/torch/fx/_lazy_graph_module.py class GraphModule(torch.nn.Module):
V0821 12:05:44.479000 371515 site-packages/torch/_dynamo/output_graph.py:2417] [0/0] [__graph_code]     def forward(self, L_x_: "f32[8][1]cuda:0"):
V0821 12:05:44.479000 371515 site-packages/torch/_dynamo/output_graph.py:2417] [0/0] [__graph_code]         l_x_ = L_x_
V0821 12:05:44.479000 371515 site-packages/torch/_dynamo/output_graph.py:2417] [0/0] [__graph_code] 
V0821 12:05:44.479000 371515 site-packages/torch/_dynamo/output_graph.py:2417] [0/0] [__graph_code]         # File: /tmp/

## 5. Graph breaks

A **graph break** happens when Dynamo hits something it can't safely trace
into the graph -- an unsupported Python builtin, a call into a C extension
it doesn't understand, `print()` on a tensor value, control flow that
depends on a runtime tensor value it can't resolve symbolically, or
`.item()` (which pulls a single value out of a GPU tensor and into a
plain Python number -- doing that requires physically waiting for every
GPU operation queued before it to actually finish and copying the result
to the CPU, a **GPU-to-CPU sync**, since the CPU can't read a value that
might not exist yet). Dynamo doesn't fail: it compiles everything
*up to* the break as one graph, runs the untraceable part in plain eager
Python, then starts a *new* graph after it. More graph breaks means less
of your function benefits from fusion/codegen, i.e. performance closer to
plain eager.

In [9]:
def has_a_graph_break(x):
    y = x * 2
    print("intermediate sum:", y.sum().item())  # forces a CPU sync + graph break
    z = y + 1
    return z.relu()

dynamo.reset()
explanation = dynamo.explain(has_a_graph_break)(torch.randn(8, device=device))
print(explanation)

/home/user/anaconda3/envs/pe_test/lib/python3.12/site-packages/torch/_dynamo/pgo.py:538: UserWarning: dynamo_pgo force disabled by torch.compiler.config.force_disable_caches
  warn_once(
W0821 12:05:44.553000 371515 site-packages/torch/_dynamo/variables/tensor.py:1379] [0/0] Graph break from `Tensor.item()`, consider setting:
W0821 12:05:44.553000 371515 site-packages/torch/_dynamo/variables/tensor.py:1379] [0/0]     torch._dynamo.config.capture_scalar_outputs = True
W0821 12:05:44.553000 371515 site-packages/torch/_dynamo/variables/tensor.py:1379] [0/0] or:
W0821 12:05:44.553000 371515 site-packages/torch/_dynamo/variables/tensor.py:1379] [0/0]     env TORCHDYNAMO_CAPTURE_SCALAR_OUTPUTS=1
W0821 12:05:44.553000 371515 site-packages/torch/_dynamo/variables/tensor.py:1379] [0/0] to include these operations in the captured graph.
W0821 12:05:44.553000 371515 site-packages/torch/_dynamo/variables/tensor.py:1379] [0/0] 
W0821 12:05:44.553000 371515 site-packages/torch/_dynamo/variables/tens

intermediate sum: 11.429878234863281
Graph Count: 2
Graph Break Count: 1
Op Count: 2
Break Reasons:
  Break Reason 1:
    Reason: Unsupported Tensor.item() call with capture_scalar_outputs=False
  Explanation: Dynamo does not support tracing `Tensor.item()` with config.capture_scalar_outputs=False.
  Hint: Set `torch._dynamo.config.capture_scalar_outputs = True` or `export TORCHDYNAMO_CAPTURE_SCALAR_OUTPUTS=1` to include these operations in the captured graph.

  Developer debug context: call_method TensorVariable() item () {}

 For more details about this graph break, please visit: https://meta-pytorch.github.io/compile-graph-break-site/gb/gb0124.html
    User Stack:
      <FrameSummary file /tmp/ipykernel_371515/1409731743.py, line 3 in has_a_graph_break>
Ops per Graph:
  Ops 1:
    <built-in function mul>
  Ops 2:
    <built-in function add>
Out Guards:
  Guard 1:
    Name: ''
    Source: shape_env
    Create Function: SHAPE_ENV
    Guard Types: None
    Code List: None
    Object W

`graph_count` should now be 2 (one before the break, one after), and
`graph_break_count` should be 1 -- with a human-readable reason attached.
The real cost isn't just "two graphs instead of one" -- it's a full stall:
kernel launch, wait for the GPU-to-CPU sync that `.item()` forces, run
plain Python (`print`), then a second kernel launch. That stall is
invisible if you only look at graph *counts*, but it's exactly why graph
breaks are expensive in practice, not just "slightly less optimal."

`fullgraph=True` turns graph breaks into hard errors instead of silent
eager fallback. This is the right setting when you specifically need the
whole function traced as one graph (e.g. for further compiler passes, or
to guarantee you're not silently leaving perf on the table).

In [10]:
dynamo.reset()
try:
    torch.compile(has_a_graph_break, fullgraph=True)(torch.randn(8, device=device))
except Exception as e:
    print(type(e).__name__, "-", str(e).splitlines()[0])

Unsupported - Failed to trace builtin operator


## 6. Guards & recompilation

A compiled graph is only valid under the assumptions Dynamo made while
tracing it -- a given input shape, dtype, device, whether a value is a
tensor vs a Python int, etc. Dynamo records those assumptions as
**guards**: cheap runtime checks re-evaluated on every call. The control
flow on every call looks like:

```
             +-------------+  guards pass  +------------------+
  call ----> | guard check |-------------->| run cached graph |
             +------+------+               +------------------+
                    | guards fail
                    v
             +------------------------------+
             | recompile: retrace through    |
             | Dynamo -> AOTAutograd ->       |
             | decomp -> Inductor (slow path) |
             +---------------+----------------+
                              v
                  cache (new graph, new guards)
```

This is why the *first* call with a new input shape is slow (compiling)
while subsequent calls with the *same* shape are fast (guards pass, cache
hit) -- and why changing shapes on every call can make `torch.compile` net
*slower* than eager, since you pay compilation cost repeatedly instead of
ever reaching the fast path.

In [11]:
torch._logging.set_logs(recompiles=True)

dynamo.reset()

def h(x):
    return x * 2 + 1

compiled_h = torch.compile(h)
compiled_h(torch.randn(4, device=device))    # compiles for shape (4,)
compiled_h(torch.randn(4, device=device))    # guards pass -> cache hit, no log
compiled_h(torch.randn(8, device=device))    # shape guard fails -> recompiles
compiled_h(torch.randn(8, 8, device=device)) # rank changes -> recompiles again

torch._logging.set_logs()

V0821 12:05:44.907000 371515 site-packages/torch/_dynamo/guards.py:4760] [0/1] [__recompiles] Recompiling function h in /tmp/ipykernel_371515/1031039714.py:5
V0821 12:05:44.907000 371515 site-packages/torch/_dynamo/guards.py:4760] [0/1] [__recompiles]     triggered by the following guard failure(s):
V0821 12:05:44.907000 371515 site-packages/torch/_dynamo/guards.py:4760] [0/1] [__recompiles]     - 0/0: tensor 'x' size mismatch at index 0. expected 4, actual 8


V0821 12:05:45.173000 371515 site-packages/torch/_dynamo/guards.py:4760] [0/2] [__recompiles] Recompiling function h in /tmp/ipykernel_371515/1031039714.py:5
V0821 12:05:45.173000 371515 site-packages/torch/_dynamo/guards.py:4760] [0/2] [__recompiles]     triggered by the following guard failure(s):
V0821 12:05:45.173000 371515 site-packages/torch/_dynamo/guards.py:4760] [0/2] [__recompiles]     - 0/1: tensor 'x' rank mismatch. expected 1, actual 2
V0821 12:05:45.173000 371515 site-packages/torch/_dynamo/guards.py:4760] [0/2] [__recompiles]     - 0/0: tensor 'x' rank mismatch. expected 1, actual 2


Two mitigations when varying shapes are expected:

- `torch._dynamo.mark_dynamic(tensor, dim)` -- tell Dynamo up front that a
  dimension is dynamic, so it traces with a symbolic size (a `SymInt`,
  i.e. a size Dynamo treats as an unknown variable rather than a fixed
  number, guarded on with an inequality like "size >= 0" instead of an
  exact-match check) instead of a guard-per-value.
- `torch.compile(fn, dynamic=True)` -- let Dynamo infer dynamism
  automatically (by default it starts static and generalizes reactively
  after it *sees* a shape change, which is what triggered the recompile
  above).

Let's confirm `dynamic=True` avoids the repeated recompiles from changing
batch size:

In [12]:
torch._logging.set_logs(recompiles=True)
dynamo.reset()

compiled_dynamic = torch.compile(h, dynamic=True)
for n in (4, 8, 16, 32):
    compiled_dynamic(torch.randn(n, device=device))

torch._logging.set_logs()
print("done: if dynamic=True worked, no [__recompiles] lines were printed above")

/home/user/anaconda3/envs/pe_test/lib/python3.12/site-packages/torch/_dynamo/pgo.py:538: UserWarning: dynamo_pgo force disabled by torch.compiler.config.force_disable_caches
  warn_once(


done: if dynamic=True worked, no [__recompiles] lines were printed above


## TorchInductor: reading generated Triton/C++ kernels

TorchInductor ([dev-discuss announcement: TorchInductor, a PyTorch-native
compiler with define-by-run IR and symbolic
shapes](https://dev-discuss.pytorch.org/t/torchinductor-a-pytorch-native-compiler-with-define-by-run-ir-and-symbolic-shapes/747))
is the default `torch.compile` backend. It takes the decomposed FX graph,
lowers it into a loop-level scheduling IR (a representation of the
computation as nested loops over indices -- this is what makes fusion
analysis tractable: two ops fuse if their loop nests can be merged into
one), decides which ops can be **fused** into a single kernel (elementwise
chains, a reduction feeding an elementwise consumer, etc.), and generates
real source code: Triton for CUDA, C++/OpenMP for CPU. That generated
source is then compiled by the real Triton/C++ toolchain and cached on
disk.

`TORCH_LOGS=output_code` (or `torch._logging.set_logs(output_code=True)`)
dumps the generated source via the logger, timestamped line-by-line --
functional but noisy to read. `torch._inductor.utils.run_and_get_triton_code`
runs the compiled function and hands back the exact same generated source
as a plain string, which is much easier to actually read. Let's do that
for our original toy fusion chain.

In [17]:
from torch._inductor.utils import run_and_get_triton_code, run_and_get_cpp_code

def elementwise_chain2(x):
    a = x * 2
    b = a + 1
    c = torch.sin(b)
    d = c * a
    return d.relu()

dynamo.reset()
compiled_elementwise_chain2 = torch.compile(elementwise_chain2)
get_code = run_and_get_triton_code if device == "cuda" else run_and_get_cpp_code
kernel_src = get_code(compiled_elementwise_chain2, torch.randn(1024, device=device))
print(kernel_src)

/home/user/anaconda3/envs/pe_test/lib/python3.12/site-packages/torch/_dynamo/pgo.py:538: UserWarning: dynamo_pgo force disabled by torch.compiler.config.force_disable_caches
  warn_once(


# AOT ID: ['8_inference']
from ctypes import c_void_p, c_long, c_int
import torch
import math
import random
import os
import tempfile
from math import inf, nan
from cmath import nanj
from torch._inductor.hooks import run_intermediate_hooks
from torch._inductor.utils import maybe_profile
from torch._inductor.codegen.memory_planning import _align as align
from torch import device, empty_strided
from torch._inductor.async_compile import AsyncCompile
from torch._inductor.select_algorithm import extern_kernels
import triton
import triton.language as tl
from torch._inductor.runtime.triton_heuristics import start_graph, end_graph
from torch._C import _cuda_getCurrentRawStream as get_raw_stream

aten = torch.ops.aten
inductor_ops = torch.ops.inductor
_quantized = torch.ops._quantized
assert_size_stride = torch._C._dynamo.guards.assert_size_stride
assert_alignment = torch._C._dynamo.guards.assert_alignment
empty_strided_cpu = torch._C._dynamo.guards._empty_strided_cpu
empty_strided_cpu_pinned =

On CUDA, look for a single `@triton.jit`-decorated kernel (often named
something like `triton_poi_fused_...`) whose body contains `tl.load` once
at the top, the whole `mul`/`add`/`sin`/`mul`/`relu` arithmetic chain in
registers, and `tl.store` once at the bottom. **One load, one store, five
ops in between** -- that's the fusion this whole pipeline exists to
deliver, and it's the same principle (minimize HBM round-trips, keep
intermediate values on-chip) that Flash Attention applies to attention
specifically. On CPU you'll see a vectorized C++/OpenMP loop instead,
doing the same fusion.

The generated `.py` wrapper (also in that dump) shows how the kernel gets
invoked -- allocate the output, launch the kernel with a computed grid,
return it -- no autograd engine, no dispatcher overhead per op, just the
kernel launch(es) TorchInductor decided were necessary.

Fusion isn't a fixed "these ops fuse, those don't" table -- it's a
scheduling decision based on dependencies, iteration structure, shapes,
and profitability. PyTorch's own tuning guide says Inductor supports
"advanced fusion involving eligible pointwise and reduction operations,"
so a reduction (an op like `sum` whose output has a different, smaller
shape than its input) doesn't automatically mean a separate kernel for
"the reduction" vs. "the elementwise ops around it." Let's build an
example with a reduction sandwiched between two elementwise ops and see
what Inductor actually does with it, rather than assuming:

In [18]:
def two_kernels(x):
    a = x * 2                 # elementwise
    b = a.sum(dim=0)          # reduction: output shape shrinks from (512,512) to (512,)
    c = torch.sin(b) * 3      # elementwise on the *reduced* tensor
    return c

dynamo.reset()
compiled_two_kernels = torch.compile(two_kernels)
kernel_src_2 = get_code(compiled_two_kernels, torch.randn(512, 512, device=device))

n_triton_kernels = kernel_src_2.count("@triton.jit") if device == "cuda" else kernel_src_2.count("kernel(")
print(f"kernel definitions found: {n_triton_kernels}")
print(kernel_src_2)

/home/user/anaconda3/envs/pe_test/lib/python3.12/site-packages/torch/_dynamo/pgo.py:538: UserWarning: dynamo_pgo force disabled by torch.compiler.config.force_disable_caches
  warn_once(


kernel definitions found: 2
# AOT ID: ['9_inference']
from ctypes import c_void_p, c_long, c_int
import torch
import math
import random
import os
import tempfile
from math import inf, nan
from cmath import nanj
from torch._inductor.hooks import run_intermediate_hooks
from torch._inductor.utils import maybe_profile
from torch._inductor.codegen.memory_planning import _align as align
from torch import device, empty_strided
from torch._inductor.async_compile import AsyncCompile
from torch._inductor.select_algorithm import extern_kernels
import triton
import triton.language as tl
from torch._inductor.runtime.triton_heuristics import start_graph, end_graph
from torch._C import _cuda_getCurrentRawStream as get_raw_stream

aten = torch.ops.aten
inductor_ops = torch.ops.inductor
_quantized = torch.ops._quantized
assert_size_stride = torch._C._dynamo.guards.assert_size_stride
assert_alignment = torch._C._dynamo.guards.assert_alignment
empty_strided_cpu = torch._C._dynamo.guards._empty_strided_cp

In [19]:
import re

if device == "cuda":
    kernel_names = re.findall(r"def (triton_\w+)\(", kernel_src_2)
    fused_ops_per_kernel = re.findall(r"Original ATen: \[([^\]]+)\]", kernel_src_2)
    print("kernel names:", kernel_names)
    print("ops attributed to each kernel (from the generated comments, de-duplicated):")
    for ops in dict.fromkeys(fused_ops_per_kernel):
        print(" ", ops)
else:
    print("kernel-name/ATen-op inspection below is CUDA-specific (Triton kernel naming); skipping on CPU.")

kernel names: ['triton_red_fused_mul_sum_0', 'triton_per_fused_mul_sin_sum_1']
ops attributed to each kernel (from the generated comments, de-duplicated):
  aten.mul, aten.sum
  aten.mul, aten.sum, aten.sin


Look at the kernel *names* Inductor generated: `triton_red_fused_mul_sum_0`
and `triton_per_fused_mul_sin_sum_1` -- both include `mul` (Section 2's
elementwise op) right alongside `sum`, and the second also includes `sin`.
The elementwise ops on **both sides** of the reduction actually got fused
directly into the reduction's own kernels; they aren't sitting in
separate "elementwise-only" kernels at all.

So why two kernels, if the pointwise ops did fuse in? Because summing 512
elements per output is itself split into two passes here for parallelism:
`triton_red_...` (a "reduction" kernel) computes partial sums over chunks
(with `x * 2` folded in as each chunk is read), and `triton_per_...` (a
"persistent" kernel) combines those partial sums into the final result
(with `sin(...) * 3` folded in once the sum is done, since they only need
the *finished* value). That's Inductor's own reduction-splitting strategy
for a reduction large enough to benefit from two passes -- not evidence
that pointwise ops can never share a kernel with a reduction. The general
rule is closer to "Inductor fuses eligible pointwise and reduction
operations when the resulting kernel is legal and likely to be faster,"
per PyTorch's own tuning guide, than a hard "reduction = kernel boundary."

That distinction matters directly for what's coming next: attention's
`softmax` is a reduction sitting *between* two matmuls (`Q @ K.T` and
`... @ V`). Section 13 covers what PyTorch actually does about that today
-- both the production answer (a pre-built fused kernel) and why this
series still implements Flash Attention by hand.

Not every op gets the Triton-codegen treatment, though. Section 3's
architecture diagram claimed Inductor sometimes calls an existing
"extern kernel" instead of generating one -- worth confirming directly
rather than taking it on faith, since it's also the concrete answer to
"is PyTorch just a wrapper around Triton?" (no). Compile a plain matmul
and check what Inductor actually emits for it:

In [20]:
def matmul_fn(x, y):
    return x @ y

dynamo.reset()
compiled_matmul = torch.compile(matmul_fn)
matmul_src = get_code(
    compiled_matmul, torch.randn(512, 512, device=device), torch.randn(512, 512, device=device)
)

if device == "cuda":
    print("generated a new @triton.jit kernel for the matmul itself:", "@triton.jit" in matmul_src)
    print("called an existing kernel instead (extern_kernels.mm):", "extern_kernels.mm" in matmul_src)
else:
    print("extern-kernel naming below is CUDA-specific (cuBLAS); skipping the same check on CPU.")
print(matmul_src)

generated a new @triton.jit kernel for the matmul itself: False
called an existing kernel instead (extern_kernels.mm): True
# AOT ID: ['10_inference']
from ctypes import c_void_p, c_long, c_int
import torch
import math
import random
import os
import tempfile
from math import inf, nan
from cmath import nanj
from torch._inductor.hooks import run_intermediate_hooks
from torch._inductor.utils import maybe_profile
from torch._inductor.codegen.memory_planning import _align as align
from torch import device, empty_strided
from torch._inductor.async_compile import AsyncCompile
from torch._inductor.select_algorithm import extern_kernels

aten = torch.ops.aten
inductor_ops = torch.ops.inductor
_quantized = torch.ops._quantized
assert_size_stride = torch._C._dynamo.guards.assert_size_stride
assert_alignment = torch._C._dynamo.guards.assert_alignment
empty_strided_cpu = torch._C._dynamo.guards._empty_strided_cpu
empty_strided_cpu_pinned = torch._C._dynamo.guards._empty_strided_cpu_pinned
empty_str

/home/user/anaconda3/envs/pe_test/lib/python3.12/site-packages/torch/_dynamo/pgo.py:538: UserWarning: dynamo_pgo force disabled by torch.compiler.config.force_disable_caches
  warn_once(


## Compilation modes

`torch.compile(fn, mode=...)` trades compile time for runtime speed:

| mode | what it adds | when to use |
|---|---|---|
| `"default"` | standard Inductor fusion | general use |
| `"reduce-overhead"` | wraps execution in **CUDA Graphs** -- a mechanism that records an entire sequence of kernel launches once ("captures" it) and replays that recording on later calls, skipping the normal per-launch setup ([NVIDIA: Getting Started with CUDA Graphs](https://developer.nvidia.com/blog/cuda-graphs/), [PyTorch: Accelerating PyTorch with CUDA Graphs](https://pytorch.org/blog/accelerating-pytorch-with-cuda-graphs/)) | many small/fast kernels where launch overhead dominates (common for inference with small batches) |
| `"max-autotune"` | benchmarks multiple kernel implementations (e.g. several Triton matmul tiling configs, or cuBLAS/cuDNN alternatives) per-op and picks the fastest measured on *your* GPU | compute-bound workloads (matmuls, convs) where you can afford much longer compile time for the best runtime |

`reduce-overhead`'s CUDA Graphs mean **captured memory addresses are
replayed** -- inputs must occupy the same memory each call, which is why it
plays awkwardly with changing-shape inputs or side effects the graph can't
see (this is a real usability tradeoff, not just a knob).

`max-autotune` can take dramatically longer to compile because it's
literally benchmarking candidate kernels against each other, not just
generating one deterministic lowering.

Let's benchmark all three on something more autotune-friendly than our
elementwise toy: a chain of matmuls.

**Expect scary-looking output from the `max-autotune` case below, and
that's normal.** Autotuning works by generating many candidate kernel
configs and trying each one; some candidates request more shared memory
than your specific GPU has available for this kernel and fail to even
compile. Triton/Inductor report each failure as a full
`RuntimeError: No valid triton configs. OutOfMemoryError: ...` traceback
at `ERROR` level, one per failed candidate -- that's the search process
working as intended (ruling out configs that don't fit), not a bug in
your setup. The actual result is in the `Autotune Choices Stats` /
`AUTOTUNE mm(...)` lines that print after all the candidates have been
tried, described below.

In [21]:
def mm_chain(x, w1, w2):
    return (x @ w1).relu() @ w2

N = 1024
x_mm = torch.randn(N, N, device=device)
w1 = torch.randn(N, N, device=device)
w2 = torch.randn(N, N, device=device)

results = {}
for mode in (None, "reduce-overhead", "max-autotune"):
    dynamo.reset()
    label = mode or "default"
    fn = torch.compile(mm_chain, mode=mode) if mode else torch.compile(mm_chain)
    t_compile0 = time.perf_counter()
    fn(x_mm, w1, w2)  # trigger compilation
    if device == "cuda":
        torch.cuda.synchronize()
    compile_time = time.perf_counter() - t_compile0

    # reduce-overhead (CUDA Graphs) needs a few extra calls beyond the first
    # to actually record the graph -- without this warmup, capture cost
    # leaks into the "steady-state" timing below and makes it look slower.
    for _ in range(5):
        fn(x_mm, w1, w2)
    if device == "cuda":
        torch.cuda.synchronize()

    us = bench(lambda x: fn(x, w1, w2), x_mm, iters=50)
    results[label] = (compile_time, us)
    print(f"{label:16s}  first-call(compile) = {compile_time:6.2f}s   steady-state = {us:8.1f} us/call")

/home/user/anaconda3/envs/pe_test/lib/python3.12/site-packages/torch/_dynamo/pgo.py:538: UserWarning: dynamo_pgo force disabled by torch.compiler.config.force_disable_caches
  warn_once(


default           first-call(compile) =   0.34s   steady-state =    192.7 us/call


/home/user/anaconda3/envs/pe_test/lib/python3.12/site-packages/torch/_dynamo/pgo.py:538: UserWarning: dynamo_pgo force disabled by torch.compiler.config.force_disable_caches
  warn_once(


reduce-overhead   first-call(compile) =   0.33s   steady-state =    228.3 us/call


/home/user/anaconda3/envs/pe_test/lib/python3.12/site-packages/torch/_dynamo/pgo.py:538: UserWarning: dynamo_pgo force disabled by torch.compiler.config.force_disable_caches
  warn_once(


...

E0821 12:05:50.717000 371515 site-packages/torch/_inductor/select_algorithm.py:3924] [0/0] Runtime error during autotuning: 
E0821 12:05:50.717000 371515 site-packages/torch/_inductor/select_algorithm.py:3924] [0/0] No valid triton configs. OutOfMemoryError: out of resource: triton_mm Required: 131072 Hardware limit:101376 Reducing block sizes or `num_stages` may help.. 
E0821 12:05:50.717000 371515 site-packages/torch/_inductor/select_algorithm.py:3924] [0/0] Ignoring this choice.


E0821 12:05:50.720000 371515 site-packages/torch/_inductor/select_algorithm.py:3924] [0/0] Runtime error during autotuning: 
E0821 12:05:50.720000 371515 site-packages/torch/_inductor/select_algorithm.py:3924] [0/0] No valid triton configs. OutOfMemoryError: out of resource: triton_mm Required: 262144 Hardware limit:101376 Reducing block sizes or `num_stages` may help.. 
E0821 12:05:50.720000 371515 site-packages/torch/_inductor/select_algorithm.py:3924] [0/0] Ignoring this choice.


E0821 12:05:50.723000 371515 site-packages/torch/_inductor/select_algorithm.py:3924] [0/0] Runtime error during autotuning: 
E0821 12:05:50.723000 371515 site-packages/torch/_inductor/select_algorithm.py:3924] [0/0] No valid triton configs. OutOfMemoryError: out of resource: triton_mm Required: 393216 Hardware limit:101376 Reducing block sizes or `num_stages` may help.. 
E0821 12:05:50.723000 371515 site-packages/torch/_inductor/select_algorithm.py:3924] [0/0] Ignoring this choice.


Autotune Choices Stats:
{"num_choices": 21, "num_triton_choices": 20, "best_kernel": "mm", "best_time": 0.10096000134944916, "best_triton_pos": 1, "best_triton_time": 0.11046399921178818, "best_triton_kernel": "triton_mm_13", "best_triton_kernel_desc": "ACC_TYPE='tl.float32', ALLOW_TF32=False, BLOCK_K=32, BLOCK_M=128, BLOCK_N=64, EVEN_K=True, GROUP_M=8, USE_FAST_ACCUM=False, num_stages=3, num_warps=4"}
AUTOTUNE mm(1024x1024, 1024x1024)
strides: [1024, 1], [1024, 1]
dtypes: torch.float32, torch.float32
  mm 0.1010 ms 100.0% 
  triton_mm_13 0.1105 ms 91.4% ACC_TYPE='tl.float32', ALLOW_TF32=False, BLOCK_K=32, BLOCK_M=128, BLOCK_N=64, EVEN_K=True, GROUP_M=8, USE_FAST_ACCUM=False, num_stages=3, num_warps=4
  triton_mm_14 0.1207 ms 83.6% ACC_TYPE='tl.float32', ALLOW_TF32=False, BLOCK_K=32, BLOCK_M=128, BLOCK_N=64, EVEN_K=True, GROUP_M=8, USE_FAST_ACCUM=False, num_stages=4, num_warps=8
  triton_mm_9 0.1228 ms 82.2% ACC_TYPE='tl.float32', ALLOW_TF32=False, BLOCK_K=32, BLOCK_M=64, BLOCK_N=128, 

max-autotune      first-call(compile) =   3.60s   steady-state =    202.1 us/call


Expect `max-autotune` to have by far the longest first-call (compile) time
-- it's literally benchmarking multiple candidate matmul kernel configs on
your GPU and keeping the winner (scan the log above for `AUTOTUNE mm(...)`
to see the candidates it tried and how each scored). On *this* workload,
don't be surprised if its steady-state time isn't clearly the fastest of
the three: `default` already dispatches plain matmuls to cuBLAS via the
same **extern kernel** mechanism verified directly in Section 9 (a call
into an existing pre-compiled library function rather than newly
generated/compiled source) -- extern kernels are one of the "candidates"
`max-autotune` benchmarks against its own generated Triton kernels.
cuBLAS is heavily hand-tuned, so autotune's own search may conclude
that same cuBLAS `mm` *is* the best option (check `"best_kernel"` in the
`Autotune Choices Stats` line above) -- in which case all three modes end
up running essentially the same matmul kernel, and any remaining
microsecond differences are timing noise from this single 50-iteration
run rather than a real effect. Where `max-autotune` earns its long compile
time is workloads with more exotic shapes/fusions where no existing
hand-tuned kernel is a good fit and Inductor's generated Triton candidates
can actually beat the generic fallback.

`reduce-overhead`'s benefit is largest when the workload is
launch-overhead-bound (many small/fast kernels) rather than compute-bound
(one big matmul chain, as here) -- on this workload its edge over
`default` is expected to be small, now that the CUDA Graph capture warmup
above has kept that one-time cost out of the timed loop.

The typical *shape* of this tradeoff -- illustrative only, read the actual
printed numbers above for what happened on your run, since they vary by
workload and GPU:

```
                  compile time                     steady-state time
  default         ▓░░░░░░░░░░░░░░░░░░░░            ▓▓▓▓▓▓▓▓▓░░░░░░░░░░
  reduce-overhead ▓▓░░░░░░░░░░░░░░░░░░░            ▓▓▓▓▓▓▓░░░░░░░░░░░░
  max-autotune    ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓            ▓▓▓▓▓▓▓░░░░░░░░░░░░
                  (benchmarks several candidate                        (often ~= default here --
                   kernel configs on YOUR GPU)                          see the explanation above)
```